# Step 07 – Class-specific errors (crops/grass vs. forest)

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 3.2 · Table 5
- **Needs:** steps 03 and 04
- **Writes:** `RESULTS_DIR/class_errors/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / class_errors/
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / train_test_gap/

In [ ]:
"""
Class-specific error analysis (focus: crops/grass and its confusion with forest)

Run AFTER step 04 (04_model_tuning_and_evaluation): it reads test_predictions.csv and train_cv_test_full.csv.

Produces, for the three models with the highest test OA:
  1. Confusion matrices (counts and row-normalised) + side-by-side figure
  2. Per-class precision (user's accuracy), recall (producer's accuracy), F1
  3. Where crops/grass errors go, and where forest errors go
  4. Whether the SAME crops/grass points are misclassified by all three models
     (shared errors point to spectral/label ambiguity rather than one algorithm)
  5. Predictor values of crops/grass points misclassified as forest vs. correctly
     classified crops/grass and forest (descriptive: medians and IQR)
  6. Optional: forest errors by forest subclass (closed/open), if the column exists
"""


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'
OUT_DIR    = RESULTS_DIR + 'class_errors/'
LABEL_COL  = 'class'

# Inferred from class counts (0: 300, 1: 950, 2: 650, 3: 430, 4: 470) - confirm against your GEE labels
CLASS_NAMES = {0: 'Water', 1: 'Forest', 2: 'Bare ground', 3: 'Urban', 4: 'Crops/grass'}
FOREST, CROPS = 'Forest', 'Crops/grass'
SUBCLASS_COL = None     # e.g. 'forest_type' if the file still holds Closed/Open forest labels
# Fixed list (not 'top 3 by test OA'): XGBoost, CatBoost and MLP tie at 95.95%, so an
# automatic ranking would pick the three in arbitrary order. These are the three leading
# models named by the reviewer, also used in the spatial CV per-class table.
TOP_MODELS = ['CatBoost', 'XGBoost', 'GTB']   # first = reference model for section 5

os.makedirs(OUT_DIR, exist_ok=True)
ORDER = list(CLASS_NAMES.values())
to_name = lambda s: s.map(lambda v: CLASS_NAMES.get(v, str(v)))

# ------------------------------------------------------------------ inputs
pred = pd.read_csv(MODEL_DIR + 'test_predictions.csv')
summary = pd.read_csv(MODEL_DIR + 'train_cv_test_full.csv')
top = TOP_MODELS
print('Models analysed (test OA %):',
      {m: round(100 * summary.set_index('Model').loc[m, 'Test_OA'], 2) for m in top})

yt = to_name(pred['y_true'])
yp = {m: to_name(pred[m]) for m in top}
assert set(yt.unique()) <= set(ORDER), f'Unmapped labels: {set(yt.unique()) - set(ORDER)} - edit CLASS_NAMES'
print('Test counts:', yt.value_counts().reindex(ORDER).to_dict())

# ------------------------------------------------------------------ 1-2. confusion matrices and per-class metrics
rows, cms = [], {}
for m in top:
    cm = confusion_matrix(yt, yp[m], labels=ORDER)
    cms[m] = cm
    pd.DataFrame(cm, index=[f'true {c}' for c in ORDER], columns=[f'pred {c}' for c in ORDER]) \
      .to_csv(OUT_DIR + f'confusion_{m}.csv')
    p, r, f, s = precision_recall_fscore_support(yt, yp[m], labels=ORDER, zero_division=0)
    for c, pp, rr, ff, ss in zip(ORDER, p, r, f, s):
        rows.append({'Model': m, 'Class': c, 'n_test': int(ss),
                     'Precision_UA (%)': 100 * pp, 'Recall_PA (%)': 100 * rr, 'F1': ff,
                     'Omission (%)': 100 * (1 - rr), 'Commission (%)': 100 * (1 - pp)})
per_class = pd.DataFrame(rows).round(2)
per_class.to_csv(OUT_DIR + 'per_class_metrics_top3.csv', index=False)
print('\n=== Per-class metrics ===')
print(per_class.pivot(index='Class', columns='Model', values='Recall_PA (%)').reindex(ORDER)
      .add_suffix(' recall').to_string())
print(per_class.pivot(index='Class', columns='Model', values='Precision_UA (%)').reindex(ORDER)
      .add_suffix(' precision').to_string())

# ------------------------------------------------------------------ 3. where do the errors go?
flows = []
for m in top:
    cm, iC, iF = cms[m], ORDER.index(CROPS), ORDER.index(FOREST)
    for src in [CROPS, FOREST]:
        i = ORDER.index(src)
        n_err = cm[i].sum() - cm[i, i]
        for j, dst in enumerate(ORDER):
            if j != i and cm[i, j] > 0:
                flows.append({'Model': m, 'True': src, 'Predicted': dst, 'n': int(cm[i, j]),
                              '% of class': round(100 * cm[i, j] / cm[i].sum(), 2),
                              '% of its errors': round(100 * cm[i, j] / n_err, 1)})
flows = pd.DataFrame(flows)
flows.to_csv(OUT_DIR + 'error_flows_crops_forest.csv', index=False)
print('\n=== Where crops/grass and forest errors go ===\n', flows.to_string(index=False))

# ------------------------------------------------------------------ 4. shared errors across the top models
is_crops = (yt == CROPS).values
wrong = np.array([(yp[m] != yt).values for m in top])
as_forest = np.array([(yp[m] == FOREST).values for m in top])
shared = {
    'crops/grass test points': int(is_crops.sum()),
    'misclassified by >= 1 model': int((wrong.any(0) & is_crops).sum()),
    'misclassified by all top models': int((wrong.all(0) & is_crops).sum()),
    'predicted forest by all top models': int((as_forest.all(0) & is_crops).sum()),
}
pd.Series(shared).to_csv(OUT_DIR + 'shared_crops_errors.csv', header=['n'])
print('\n=== Shared crops/grass errors ===')
for k, v in shared.items():
    print(f'  {k:38s}: {v}')

# ------------------------------------------------------------------ 5. predictor values of the confused points
PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()
df = pd.read_csv(DATA_PATH)
feats = df.loc[pred['row_idx'].values, PREDICTORS].reset_index(drop=True)
best = top[0]
groups = pd.Series(np.select(
    [(yt == CROPS) & (yp[best] == FOREST), (yt == CROPS) & (yp[best] == CROPS),
     (yt == FOREST) & (yp[best] == CROPS), (yt == FOREST) & (yp[best] == FOREST)],
    ['Crops→Forest', 'Crops correct', 'Forest→Crops', 'Forest correct'], default='other'))
g = feats[groups != 'other'].groupby(groups[groups != 'other'])
med = g.median().T
iqr = (g.quantile(0.75) - g.quantile(0.25)).T
n_grp = groups.value_counts()
desc = pd.concat({'median': med, 'IQR': iqr}, axis=1).round(4)
desc.to_csv(OUT_DIR + f'predictors_confused_points_{best}.csv')
print(f'\n=== Predictor medians by group ({best}) | group sizes: '
      f'{n_grp.drop("other", errors="ignore").to_dict()} ===')
print(med.round(4).to_string())

# ------------------------------------------------------------------ 6. optional: forest subclass
if SUBCLASS_COL and SUBCLASS_COL in df.columns:
    sub = df.loc[pred['row_idx'].values, SUBCLASS_COL].reset_index(drop=True)
    for m in top:
        ct = pd.crosstab(sub[yt == FOREST], yp[m][yt == FOREST])
        ct.to_csv(OUT_DIR + f'forest_subclass_{m}.csv')
        print(f'\n=== {m}: forest test points by subclass vs. prediction ===\n', ct)

# ------------------------------------------------------------------ figure: row-normalised confusion matrices
fig, axes = plt.subplots(1, len(top), figsize=(4.2 * len(top), 4.0))
for ax, m in zip(np.atleast_1d(axes), top):
    cm = cms[m]
    pct = 100 * cm / cm.sum(1, keepdims=True)
    im = ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(len(ORDER)):
        for j in range(len(ORDER)):
            ax.text(j, i, f'{cm[i, j]}\n({pct[i, j]:.1f}%)', ha='center', va='center', fontsize=7,
                    color='white' if pct[i, j] > 60 else 'black')
    ax.set_xticks(range(len(ORDER)), ORDER, rotation=40, ha='right', fontsize=8)
    ax.set_yticks(range(len(ORDER)), ORDER, fontsize=8)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_title(m, fontsize=10, fontweight='bold')
np.atleast_1d(axes)[0].set_ylabel('Reference', fontsize=9)
cbar = fig.colorbar(im, ax=axes, shrink=0.75)
cbar.set_label('% of reference class (recall)', fontsize=8)
fig.savefig(OUT_DIR + 'Fig_confusion_top3.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig_confusion_top3.pdf', bbox_inches='tight')
plt.show()
print('\nFiles saved to:', OUT_DIR)